# Arm G all-position ablation + refusal discriminant

Every Arm G intervention so far removed the conflict component at **one layer, at the
final prompt position only**, and flipped **zero** decisions in 128 rows. The published
single-direction results that *do* flip behaviour ablate across **every layer and every
token position**. So the existing null may be a fact about our intervention rather than
about the variable.

This run separates those explanations with a 2×2 over layer scope × position scope,
using the frozen rank-1 layer-16 conflict direction on fresh seed-108 prompts.

**Second question:** is the Arm G direction just a refusal/harmfulness direction under
another name? A refusal direction is built by the standard difference-of-means
construction over matched harmful/harmless instructions, then compared geometrically
(cosine) and causally (does ablating it attenuate the scope-conflict contrast?).

**Coherence guard:** action probability mass, top-token-is-action rate, next-token
entropy and KL from baseline are recorded for every condition. A decision flip produced
by breaking the model is not evidence about the variable.

Set **Runtime → Change runtime type → A100 GPU**. ~130 batched forward passes; a few
minutes plus model load.

In [ ]:
# Colab supplies torch/CUDA.
print("Protocol: ARM_G_ALLPOS_V1")
%pip -q install "transformers==5.0.0" "accelerate==1.12.0" \
  "sentence-transformers==5.2.2" "scikit-learn==1.8.0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
LAUNCH_DIR = "/content/drive/MyDrive/phi-map/arm-g-allpos-launch"
LAUNCH_FILES = (
    "arm_g_allpos.py",
    "arm_g_cross_layer.py",
    "arm_g_causal_subspace.py",
    "arm_g_causal_dose_ablation.py",
    "arm_g_causal.py",
    "arm_g_phase1.py",
    "arm_g_scenarios.py",
)
for name in LAUNCH_FILES:
    source = f"{LAUNCH_DIR}/{name}"
    assert os.path.exists(source), f"Missing {source}"
    shutil.copy2(source, f"/content/{name}")
print("Arm G all-position launch files staged: OK")

In [ ]:
ACTING_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
WORK_DIR = "/content/drive/MyDrive/phi-map/arm-g-allpos-seed108-v1"
PAIRS_PER_FAMILY = 16
BOOTSTRAP = 2000
RANDOM_DIRECTIONS = 8
BATCH_SIZE = 16

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN secret with Llama-3.1-8B-Instruct access"

In [ ]:
from huggingface_hub import hf_hub_download
hf_hub_download(ACTING_MODEL, "config.json", token=HF_TOKEN)
print("Hugging Face model access: OK")

import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")
assert "A100" in props.name and props.total_memory >= 35 * 1024**3
assert torch.cuda.is_bf16_supported()

In [ ]:
# Deterministic tests: hook position scoping and decision-flip detection.
import os, subprocess, sys
env = dict(os.environ)
env["HF_TOKEN"] = HF_TOKEN
base_cmd = [
    sys.executable, "/content/arm_g_allpos.py",
    "--model", ACTING_MODEL,
    "--output-dir", WORK_DIR,
    "--pairs-per-family", str(PAIRS_PER_FAMILY),
    "--bootstrap", str(BOOTSTRAP),
    "--random-directions", str(RANDOM_DIRECTIONS),
    "--batch-size", str(BATCH_SIZE),
]
subprocess.run(base_cmd + ["--self-test"], check=True, env=env)

In [ ]:
# 2x2 scope factorial, layer-17 variant, refusal direction, random controls.
subprocess.run(base_cmd, check=True, env=env)

In [ ]:
import json
result_path = f"{WORK_DIR}/arm_g_allpos_result.json"
result = json.load(open(result_path))
summary = {
    "headline": result["headline"],
    "baseline": result["baseline"],
    "conditions": {
        name: {
            "attenuation": e["attenuation"]["overall"],
            "attenuation_fraction": e["attenuation_fraction"],
            "ci_95": e["attenuation_bootstrap"]["ci_95"],
            "decision_flips": e["decision_flips"],
            "decision_flip_rate": e["decision_flip_rate"],
            "coherence": e["coherence"],
        }
        for name, e in result["conditions"].items()
    },
    "discriminant": result["discriminant"],
    "random_direction_controls": {
        k: v for k, v in result["random_direction_controls"].items() if k != "details"
    },
    "sample_counts": result["sample_counts"],
}
print(json.dumps(summary, indent=2))

In [ ]:
import base64, gzip, json

summary_path = f"{WORK_DIR}/arm_g_allpos_result_summary.json"
archive_path = f"{WORK_DIR}/arm_g_allpos_result.json.gz.b64"
with open(summary_path, "w") as h:
    json.dump({**summary, "protocol": {"source_seeds": [101, 102], "evaluation_seed": 108,
        "direction_layer": 16, "bootstrap_repetitions": BOOTSTRAP},
        "full_result_artifact": "arm_g_allpos_result.json.gz.b64"}, h, indent=1)
raw = json.dumps(result).encode("utf-8")
with open(archive_path, "wb") as h:
    h.write(base64.b64encode(gzip.compress(raw)))
print("summary:", summary_path)
print("archive:", archive_path)